# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | ** | 
| **Mehmet Kilinc 25135007** | *khaleelalaqel* |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [12]:
# Gedeelde imports (door iedereen te gebruiken/aan te vullen)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (6, 4)

RANDOM_STATE = 42 


**pd.set_option** zorgt ervoor dat pandas alle kolommen toont wanneer je een dataframe print, in plaats van kolommen af te korten met ...
<br>

**plt.rcParams** stelt de standaardgrootte in van elke grafiek die we met matplotlib maken

<br>

### Data inladen

In [13]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




### Bronverantwoording AI bij hoofdstukken 5.6 t/m 5.8
De eerste codeversies voor deze hoofdstukken zijn gegenereerd in één
Claude-gesprek met Claude Opus 5.5 op High effort, met de notebooks van Ahmed
en Khaleel als context. Hieronder staat de verwijzing naar
het Claude-gesprek; de vakinhoudelijke documentatie staat in de bronnenlijst
aan het einde van deze hoofdstukken.

**Vul de echte chattitel en gedeelde chatlink in deze ene verwijzing in:**

Anthropic. (2026). *Stroke Prediction ML* [AI-gesprek met Claude Opus 5.5,
High effort]. [Gesprek bekijken](https://claude.ai/share/516aef1f-94b6-4e5c-9ea4-614401855168).

Bij de code van 5.6, 5.7 en 5.8 verwijzen we naar ditzelfde gesprek als
**Anthropic (2026)**. 

In [14]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [15]:
# Imports

import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict, validation_curve
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

### 5.6 Decision Tree
Een beslisboom voorspelt op basis van opeenvolgende vragen over de features.
In dit hoofdstuk berekenen we splitscriteria, onderzoeken we pruning en tunen
en visualiseren we een classifier.

**Codebasis:** Anthropic (2026), zie de bronverantwoording AI hierboven.

#### 5.6.1 Gini, entropie en information gain
Een beslisboom splitst de patiënten steeds in twee groepen met een vraag zoals "is `age` ≤ 55?". Bij iedere knoop probeert de boom alle features en drempels. Hij kiest de split waarbij de twee groepen het **zuiverst** worden, dus zo veel mogelijk patiënten van één klasse bevatten. Die (on)zuiverheid meten we met Gini of entropie. Hierbij is $p_k$ het aandeel van klasse $k$ in de knoop.

| Maat | Formule | Zuiver (één klasse) | Maximaal onzuiver (50/50) |
|---|---|---|---|
| **Gini** | $G = 1 - \sum_k p_k^2$ | 0 | 0,5 |
| **Entropie** | $H = -\sum_k p_k \log_2 p_k$ | 0 | 1 |

De **winst** van een split is de onzuiverheid van de ouder min het gewogen gemiddelde van de twee kinderen:

$$\text{gain} = I(\text{ouder}) - \left(\frac{n_L}{n} I(L) + \frac{n_R}{n} I(R)\right)$$

Met entropie heet dit de **information gain**, met Gini de **Gini-daling**. De boom kiest de split met de hoogste winst. In scikit-learn stel je dit in met `criterion="gini"` of `criterion="entropy"`.

**Rekenvoorbeeld.** We nemen 20 patiënten uit de trainingsset, waarvan 6 met stroke. Dat is bewust 30% in plaats van 1,5%: anders zit er bijna geen stroke-patiënt in de steekproef en is er niets te splitsen. Voor deze 20 patiënten vergelijken we een split op leeftijd met een split op hoge bloeddruk.

(scikit-learn developers, z.d.-a)

In [16]:
# Eigen functies voor Gini, entropie en information gain
def class_counts(y):
    """Aantal patiënten zonder (0) en met (1) stroke."""
    y = np.asarray(y)
    return np.array([(y == 0).sum(), (y == 1).sum()])


def gini(y):
    """Gini-onzuiverheid: 1 - som van p_k²."""
    if len(y) == 0:
        return 0.0
    p = class_counts(y) / len(y)
    return float(1 - np.sum(p ** 2))


def entropy(y):
    """Entropie: -som van p_k · log2(p_k); klassen met p = 0 tellen niet mee."""
    if len(y) == 0:
        return 0.0
    p = class_counts(y) / len(y)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p))) + 0.0  # + 0.0 voorkomt "-0.000"


def information_gain(y, left_mask, impurity=entropy):
    """Onzuiverheid ouder min gewogen onzuiverheid van de twee kinderen."""
    y, left_mask = np.asarray(y), np.asarray(left_mask)
    left, right = y[left_mask], y[~left_mask]
    children = (len(left) * impurity(left)
                + len(right) * impurity(right)) / len(y)
    return impurity(y) - children


def gini_text(y):
    """Gini-berekening als tekst, bijv. 1 - (14/20)² - (6/20)²."""
    if len(y) == 0:
        return "lege groep (telt niet mee)"
    n0, n1 = class_counts(y)
    n = n0 + n1
    return f"1 - ({n0}/{n})² - ({n1}/{n})² = {gini(y):.3f}"


def entropy_text(y):
    """Entropie-berekening als tekst."""
    if len(y) == 0:
        return "lege groep (telt niet mee)"
    n0, n1 = class_counts(y)
    n = n0 + n1
    terms = " - ".join(f"({k}/{n})·log2({k}/{n})" for k in (n0, n1) if k > 0)
    return f"-{terms} = {entropy(y):.3f}"


def show_split(name, y, left_mask):
    """Toont aantallen en berekeningen van één split en geeft een tabelrij terug."""
    y, left_mask = np.asarray(y), np.asarray(left_mask)
    left, right = y[left_mask], y[~left_mask]
    n, n_l, n_r = len(y), len(left), len(right)
    print("Split:", name)
    for part, values in [("ouder", y), ("links", left), ("rechts", right)]:
        n0, n1 = class_counts(values)
        print(part, ":", len(values), "patiënten; geen stroke:", n0, "stroke:", n1)
        print("Gini:", gini_text(values))
        print("Entropie:", entropy_text(values))
    gini_gain = information_gain(y, left_mask, gini)
    info_gain = information_gain(y, left_mask, entropy)
    print("Gewichten links/rechts:", round(n_l / n, 3), "/", round(n_r / n, 3))
    print("Gini-daling =", round(gini(y), 3), "-",
          round(gini(y) - gini_gain, 3), "=", round(gini_gain, 3))
    print("Information gain =", round(entropy(y), 3), "-",
          round(entropy(y) - info_gain, 3), "=", round(info_gain, 3))
    print()
    return {"split": name, "n links": n_l, "stroke links": left.sum(),
            "n rechts": n_r, "stroke rechts": right.sum(),
            "Gini-daling": gini_gain, "information gain": info_gain}

In [17]:
# Steekproef van 20 patiënten uit de trainingsset: 6 met en 14 zonder stroke
tree_features = ["age", "hypertension"]
mini_tree = X_train[tree_features].copy()
mini_tree["stroke"] = np.asarray(y_train)
mini_tree = pd.concat([
    mini_tree[mini_tree["stroke"] == 1].sample(6, random_state=RANDOM_STATE),
    mini_tree[mini_tree["stroke"] == 0].sample(14, random_state=RANDOM_STATE),
]).reset_index(drop=True)
y_tree = mini_tree["stroke"].to_numpy()

print("Aantal patiënten:", len(y_tree))
print("Met stroke:", y_tree.sum(), "zonder stroke:", (y_tree == 0).sum())
print("Met hoge bloeddruk:", mini_tree["hypertension"].sum())
mini_tree

NameError: name 'X_train' is not defined

In [ ]:
def best_threshold(x, y, impurity=gini):
    """Zoekt de drempel met de hoogste winst (midden tussen twee waarden, net als sklearn)."""
    values = np.unique(x)
    thresholds = (values[:-1] + values[1:]) / 2
    gains = [information_gain(y, x <= t, impurity) for t in thresholds]
    best = int(np.argmax(gains))
    return thresholds[best], gains[best]


age = mini_tree["age"].to_numpy()
hypertension = mini_tree["hypertension"].to_numpy()
age_threshold, _ = best_threshold(age, y_tree, gini)

# Links = antwoord "ja" op de vraag, rechts = "nee"
split_rows = [
    show_split(f"age <= {age_threshold:.2f}", y_tree, age <= age_threshold),
    show_split("hypertension <= 0.5 (links = geen hoge bloeddruk)",
               y_tree, hypertension <= 0.5)]
pd.DataFrame(split_rows).set_index("split").round(3)

In [ ]:
# Boom met diepte 1 zonder class weights op de volledige trainingsset:
# sklearn kiest zelf de beste eerste split uit alle features.
# Daarna rekenen we dezelfde split na met onze eigen functies.
y_full_train = np.asarray(y_train)
check_rows = []
for criterion, impurity in [("gini", gini), ("entropy", entropy)]:
    stump = DecisionTreeClassifier(max_depth=1, criterion=criterion,
                                   random_state=RANDOM_STATE)
    stump.fit(X_train, y_full_train)
    tree = stump.tree_
    left, right = tree.children_left[0], tree.children_right[0]
    n = tree.n_node_samples

    # Winst volgens sklearn (uit de onzuiverheid per knoop)
    sklearn_gain = tree.impurity[0] - (n[left] * tree.impurity[left]
                                       + n[right] * tree.impurity[right]) / n[0]
    # Dezelfde split met onze eigen functies
    feature = X_train.columns[tree.feature[0]]
    left_mask = X_train[feature].to_numpy() <= tree.threshold[0]
    own_gain = information_gain(y_full_train, left_mask, impurity)

    check_rows.append({
        "criterium": criterion, "feature": feature, "drempel": tree.threshold[0],
        "n links": n[left], "n rechts": n[right],
        "ouder (sklearn)": tree.impurity[0], "ouder (eigen)": impurity(y_full_train),
        "links (sklearn)": tree.impurity[left],
        "links (eigen)": impurity(y_full_train[left_mask]),
        "rechts (sklearn)": tree.impurity[right],
        "rechts (eigen)": impurity(y_full_train[~left_mask]),
        "gain (sklearn)": sklearn_gain, "gain (eigen)": own_gain,
        "gelijk": np.isclose(sklearn_gain, own_gain)})

    if criterion == "gini":
        # De Gini-boom tekenen
        fig, ax = plt.subplots(figsize=(8, 3.5))
        plot_tree(stump, feature_names=list(X_train.columns),
                  class_names=["Geen stroke", "Stroke"],
                  filled=True, impurity=True, ax=ax)
        ax.set_title("Eerste split van sklearn op de trainingsset "
                     "(criterion = gini, diepte 1)")
        plt.show()

# 6 decimalen: op de volledige set zijn de winsten erg klein
pd.DataFrame(check_rows).set_index("criterium").round(6).T

#### 5.6.2 Pre-pruning en post-pruning
Een beslisboom zonder grenzen kan doorgaan met splitsen zolang de data dat toelaat. Op de trainingsdata scoort hij dan bijna perfect, maar hij heeft vooral de ruis geleerd (**overfitting**). **Pruning** maakt de boom kleiner, zodat hij beter generaliseert.

- **Pre-pruning** stopt de boom tijdens het groeien, bijvoorbeeld met `max_depth`: de boom mag niet dieper dan een vast aantal niveaus.
- **Post-pruning** laat de boom eerst volledig groeien en snoeit daarna takken weg die weinig opleveren. scikit-learn gebruikt hiervoor *minimal cost-complexity pruning*:

$$R_\alpha(T) = R(T) + \alpha \cdot |T|$$

Hierin is $R(T)$ de totale (gewogen) onzuiverheid van de bladeren en $|T|$ het aantal bladeren. Iedere extra blad kost $\alpha$. Een tak blijft alleen staan als hij de onzuiverheid meer verlaagt dan hij kost. Hoe groter `ccp_alpha`, hoe kleiner de boom. Het **pruning-pad** (`cost_complexity_pruning_path`) geeft alle alpha's waarbij de boom een stap kleiner wordt.

We vergelijken `max_depth` van 1 t/m 20 en ongeveer 15 alpha's uit het pruning-pad, gekozen op een log-schaal. Iedere waarde wordt getest met dezelfde 5-voudige gestratificeerde cross-validation als bij de andere modellen. We gebruiken `class_weight="balanced"` en de F1 bij de beste drempel (`F1_BEST_SCORER`). Dit gebeurt alleen op de trainingsset; de validatieset bewaren we voor de eindevaluatie.

De alpha-waarden komen uit het pruning-pad van de hele trainingsset. De CV-curves gebruiken we daarom als verkennende vergelijking; de aparte validatieset blijft buiten deze stap. De grootste alpha geeft op de hele trainingsset een boom met alleen de wortel, maar kan in een CV-fold nog een kleine boom overlaten.

(scikit-learn developers, z.d.-a, z.d.-f)

In [ ]:
# Dezelfde gestratificeerde folds als in ModelExperiment
CV_PRUNING = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                             random_state=RANDOM_STATE)
BASE_TREE = DecisionTreeClassifier(class_weight="balanced",
                                   random_state=RANDOM_STATE)


def pruning_curve(param_name, values):
    """Train- en CV-F1 (beste drempel) per waarde van één hyperparameter.

    Alleen de trainingsset wordt gebruikt; de validatieset blijft apart.
    """
    train_scores, cv_scores = validation_curve(
        BASE_TREE, X_train, y_train, param_name=param_name,
        param_range=values, scoring=F1_BEST_SCORER, cv=CV_PRUNING,
        n_jobs=N_JOBS)
    curve = pd.DataFrame({
        "train gem": train_scores.mean(axis=1),
        "train std": train_scores.std(axis=1),
        "CV gem": cv_scores.mean(axis=1),
        "CV std": cv_scores.std(axis=1)},
        index=pd.Index(values, name=param_name))
    # Score per fold, zodat de spreiding zichtbaar is
    for fold in range(cv_scores.shape[1]):
        curve[f"fold {fold + 1}"] = cv_scores[:, fold]
    return curve


def plot_pruning_curve(curve, label, log_x=False):
    """Train- en CV-F1 met band van ±1 std, losse folds en het beste punt."""
    x = curve.index.to_numpy()
    fold_columns = [col for col in curve.columns if col.startswith("fold")]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for name, color in [("train", "#4C72B0"), ("CV", "#C44E52")]:
        mean, std = curve[name + " gem"], curve[name + " std"]
        ax.plot(x, mean, marker="o", color=color, label=name + "-F1 (gemiddelde)")
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.15)
    # Iedere CV-fold als losse grijze stip
    for col in fold_columns:
        ax.scatter(x, curve[col], color="grey", s=10, alpha=0.6,
                   label="CV-F1 per fold" if col == fold_columns[0] else None)
    # Beste punt: hoogste gemiddelde CV-F1
    best = curve["CV gem"].idxmax()
    best_text = f"{best:.2e}" if log_x else str(best)
    ax.axvline(best, color="black", linestyle="--", linewidth=1)
    ax.scatter(best, curve.loc[best, "CV gem"], marker="*", s=250,
               color="gold", edgecolor="black", zorder=5,
               label=f"beste: {label} = {best_text} "
                     f"(CV-F1 {curve.loc[best, 'CV gem']:.3f})")
    if log_x:
        ax.set_xscale("log")
    ax.set(xlabel=label, ylabel="F1 bij beste drempel",
           title=f"Decision tree: train- en CV-F1 per {label} (band = ±1 std)")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()
    return best

In [ ]:
# Pre-pruning: maximale diepte 1 t/m 20
depths = list(range(1, 21))
depth_curve = pruning_curve("max_depth", depths)
best_depth = plot_pruning_curve(depth_curve, "max_depth")
depth_curve.round(3)

In [ ]:
# Post-pruning: het pruning-pad van de volledige boom op de trainingsset
path = BASE_TREE.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas[path.ccp_alphas > 1e-10]  # alpha 0 (volle boom) en afrondingsruis weg
inner_alphas, root_alpha = alphas[:-1], alphas[-1]  # grootste alpha = alleen de wortel

# 14 waarden op een log-schaal (dichtstbijzijnde alpha uit het pad) + de wortel-alpha
targets = np.geomspace(inner_alphas.min(), inner_alphas.max(), 14)
ccp_alphas = np.unique(
    [inner_alphas[np.abs(np.log(inner_alphas) - np.log(t)).argmin()] for t in targets]
    + [root_alpha])
print("Aantal alpha's in het pruning-pad:", len(path.ccp_alphas))
print("Aantal gekozen alpha's:", len(ccp_alphas))
print(f"Bereik: {ccp_alphas.min():.2e} t/m {ccp_alphas.max():.2e}")

In [ ]:
# Post-pruning: train- en CV-F1 per ccp_alpha
alpha_curve = pruning_curve("ccp_alpha", ccp_alphas)
best_alpha = plot_pruning_curve(alpha_curve, "ccp_alpha", log_x=True)
alpha_curve.round(4)

#### 5.6.3 Hyperparameter tuning
**Pipeline:** alleen `DecisionTreeClassifier`. Een boom splitst op één feature tegelijk, dus standaardiseren is niet nodig (paragraaf 3.8).

**Technieken tegen imbalance:** (1) `class_weight="balanced"`; (2) beslisdrempel afstemmen.

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `criterion` | gini, entropy | Maat voor de onzuiverheid van een knoop (paragraaf 5.6.1) |
| `max_depth` | 3 / 5 / 7 / 10 | Pre-pruning: maximale diepte, van ondiep tot iets dieper |
| `min_samples_leaf` | 1 / 20 / 50 | Pre-pruning: minimaal aantal patiënten per blad; waarden boven 1 beperken kleine bladeren |
| `ccp_alpha` | 0 / 0,0005 / 0,002 | Post-pruning: kosten per blad (paragraaf 5.6.2); 0 = niet snoeien |

Deze waarden zijn een logische eerste keuze. Ze lopen van een ondiepe, eenvoudige boom tot een iets diepere boom, met daarbij geen, lichte of sterkere pruning. Zeer diepe bomen laten we weg, omdat die de trainingsdata vooral uit het hoofd leren. Met 2 × 4 × 3 × 3 = 72 combinaties en 5 folds blijft het grid overzichtelijk. Als de pruningcurves in 5.6.2 een duidelijk ander optimum laten zien, kunnen we de grid daar alsnog op aanpassen.

`ccp_alpha` bepaalt de boete per blad bij cost-complexity pruning. Door het verschil in schaal kan dezelfde alpha-waarde bij Gini en entropie anders uitpakken. Daarom zoeken we `criterion` en `ccp_alpha` samen in één grid.

Het grid wordt beoordeeld met onze gedeelde scorer: F1 bij de beste drempel binnen iedere CV-fold. Deze CV-F1 is een selectiescore en kan optimistisch zijn. De uiteindelijke validatie-F1 gebruikt de drempel die uit de out-of-fold scores van de trainingsset is gekozen.

(scikit-learn developers, z.d.-c)

In [ ]:
# Decision tree via de vaste aanpak: tunen, drempel kiezen, valideren, Kaggle-bestand
decision_tree = ModelExperiment(
    name="Decision Tree",
    pipeline=Pipeline([
        ("model", DecisionTreeClassifier(class_weight="balanced",
                                         random_state=RANDOM_STATE))]),
    param_grid={"model__criterion": ["gini", "entropy"],
                "model__max_depth": [3, 5, 7, 10],
                "model__min_samples_leaf": [1, 20, 50],
                "model__ccp_alpha": [0.0, 0.0005, 0.002]},
    techniques="class weights + drempel").run(data)
results.add(decision_tree)

In [ ]:
decision_tree.plot_evaluation(y_val)

# Gekozen instellingen, drempel en scores in één tabel
best_tree = decision_tree.best_model.named_steps["model"]
m = decision_tree.val_metrics
dt_settings = pd.Series({
    **{key.split("__")[-1]: value
       for key, value in decision_tree.search.best_params_.items()},
    "class_weight": "balanced",
    "diepte getunede boom": best_tree.get_depth(),
    "bladeren getunede boom": best_tree.get_n_leaves(),
    "drempel": round(decision_tree.threshold, 3),
    "CV-F1": round(decision_tree.search.best_score_, 3),
    "validatie-F1 (gekozen drempel)": round(m["f1"], 3),
    "validatie-F1 (drempel 0,5)": round(m["f1_default"], 3),
    "precision": round(m["precision"], 3),
    "recall": round(m["recall"], 3),
    "TP": m["TP"], "FP": m["FP"], "FN": m["FN"],
}).to_frame("Decision Tree")
dt_settings

#### 5.6.4 Interpretatie: boom en belangrijkste features
We tekenen de bovenste drie niveaus van het beste model en tonen de tien
belangrijkste features. Bij de figuur tonen we de volledige diepte en het aantal
bladeren. Door de class weights zijn de waarden in de boom gewogen
klasseaandelen; ze zijn geen directe weergave van het echte stroke-percentage.

Feature importance geeft aan hoeveel een feature heeft bijgedragen aan de
daling in onzuiverheid. Het zegt niet of een hogere waarde het risico verhoogt
of verlaagt. Deze maat kan features met veel mogelijke splitpunten bevoordelen.

(scikit-learn developers, z.d.-a, z.d.-b)

In [ ]:
# Bovenste drie niveaus van de getunede boom (wortel + 2 niveaus splits)
best_tree = decision_tree.best_model.named_steps["model"]
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(best_tree, max_depth=2, feature_names=list(X_train.columns),
          class_names=["Geen stroke", "Stroke"], filled=True, rounded=True,
          impurity=True, proportion=True, precision=3, fontsize=10, ax=ax)
ax.set_title("Decision tree: bovenste 3 niveaus", fontsize=14)
print("Volledige diepte:", best_tree.get_depth(),
      "aantal bladeren:", best_tree.get_n_leaves())
plt.show()

In [ ]:
def top_features(model, columns, n=10):
    """De n belangrijkste features volgens feature_importances_."""
    return (pd.Series(model.feature_importances_, index=columns)
            .sort_values(ascending=False).head(n))


def plot_importances(importances, title):
    """Horizontale staafgrafiek, belangrijkste feature bovenaan."""
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(importances) + 1))
    ax.barh(importances.index[::-1], importances.values[::-1], color="#55A868")
    ax.set(title=title,
           xlabel="Feature importance (aandeel in totale daling onzuiverheid)")
    plt.tight_layout()
    plt.show()


dt_importances = top_features(best_tree, X_train.columns)
plot_importances(dt_importances, "Decision tree: tien belangrijkste features")
dt_importances.to_frame("importance").assign(
    **{"aandeel (%)": lambda t: t["importance"] * 100}).round(3)

### 5.7 Ensembles: Random Forest, Gradient Boosting en XGBoost
Bij bagging worden modellen op verschillende bootstrap-steekproeven getraind
en worden hun uitkomsten gecombineerd. Random Forest voegt hier willekeurige
featureselectie per split aan toe. Deze bomen kunnen onafhankelijk van elkaar
worden gebouwd. Bij boosting worden bomen na elkaar toegevoegd, waarbij een
nieuwe boom de fouten van het opgebouwde model probeert te verkleinen.
We vergelijken de modellen via dezelfde experimenteerfunctie en meten ook
hun trainingstijd.

(scikit-learn developers, z.d.-b)

**Codebasis:** Anthropic (2026), zie de bronverantwoording AI hierboven.

#### 5.7.1 Random Forest: hyperparameter tuning
**Pipeline:** alleen `RandomForestClassifier` met 300 bomen. Net als bij de decision tree is standaardiseren niet nodig.

**Technieken tegen imbalance:** (1) `class_weight="balanced_subsample"`: iedere boom krijgt de class weights opnieuw berekend op zijn eigen bootstrap-steekproef; (2) beslisdrempel afstemmen.

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `max_depth` | 8 / None | Een beperkte diepte of doorgroeien tot de andere grenzen |
| `min_samples_leaf` | 10 / 30 | Minimaal aantal patiënten per blad |
| `max_features` | sqrt / 0,5 | Hoeveel features per split worden bekeken: de wortel van het aantal features of ongeveer de helft |

Met weinig stroke-patiënten kunnen kleine bladeren toevallige patronen leren. Daarom vergelijken we twee minimale bladgroottes en een beperkte diepte met een boom zonder vaste dieptegrens. Het beperken van het aantal features per split helpt de bomen onderling te laten verschillen. Dit zijn logische eerste waarden; met 2 × 2 × 2 = 8 combinaties blijft het grid overzichtelijk.

Bij 5-fold cross-validation zijn dit 40 trainingen voor het grid, één refit van het beste model, vijf trainingen voor de out-of-fold drempelkeuze en één training op alle data voor het submissionbestand: 47 trainingen in totaal. De echte rekentijd hangt af van de data en de computer.

(scikit-learn developers, z.d.-b, z.d.-e)

In [ ]:
def run_experiment(name, pipeline, param_grid, techniques):
    """Voert ModelExperiment uit, voegt het toe aan results en toont de evaluatie."""
    experiment = ModelExperiment(name=name, pipeline=pipeline,
                                 param_grid=param_grid,
                                 techniques=techniques).run(data)
    results.add(experiment)
    experiment.plot_evaluation(data["y_val"])
    return experiment

In [ ]:
# Random forest: 300 bomen, class weights per bootstrap-steekproef.
# n_jobs=1 per model; de cross-validation in ModelExperiment gebruikt al alle kernen.
random_forest = run_experiment(
    name="Random Forest",
    pipeline=Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300, class_weight="balanced_subsample",
            n_jobs=1, random_state=RANDOM_STATE))]),
    param_grid={"model__max_depth": [8, None],
                "model__min_samples_leaf": [10, 30],
                "model__max_features": ["sqrt", 0.5]},
    techniques="class weights (balanced_subsample) + drempel")

#### 5.7.2 Gradient Boosting: hyperparameter tuning
**Pipeline:** `RandomUnderSampler` → `GradientBoostingClassifier` met `subsample=0.8`. Iedere nieuwe boom wordt getraind op een willekeurige 80% van de (geresamplede) trainingsdata. Dat kan het model minder gevoelig voor ruis maken (*stochastic gradient boosting*). Standaardiseren is niet nodig.

**Technieken tegen imbalance:** (1) random undersampling, omdat `GradientBoostingClassifier` in scikit-learn geen `class_weight` heeft; (2) beslisdrempel afstemmen.

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `sampling_strategy` | 0,1 / 0,25 | Verhouding stroke : gezond na undersampling (1 : 10 of 1 : 4) |
| `n_estimators` | 100 / 300 | Aantal bomen dat na elkaar wordt toegevoegd |
| `learning_rate` | 0,05 / 0,1 | Hoe sterk iedere nieuwe boom de voorspelling bijstuurt |
| `max_depth` | 2 / 3 | Diepte van iedere boom; ondiepe bomen zijn gebruikelijk bij boosting |

We beginnen met ondiepe bomen. Bij gradient boosting probeert iedere nieuwe boom de fouten van het model tot dan toe te corrigeren. De learning rate bepaalt hoe zwaar de nieuwe boom meetelt. `n_estimators` en `learning_rate` hangen samen: een kleinere learning rate vraagt doorgaans om meer bomen. Daarom tunen we ze samen. De sampler staat in de pipeline. Zo wordt alleen het trainingsdeel van iedere fold geresampled en blijven de validatiefolds en de validatieset onaangetast. Met 2 × 2 × 2 × 2 = 16 combinaties en 5 folds blijft het grid overzichtelijk.

(scikit-learn developers, z.d.-b; imbalanced-learn developers, z.d.)

In [ ]:
# Gradient boosting: geen class_weight, dus random undersampling in de pipeline.
# De sampler werkt alleen op het trainingsdeel van iedere fold.
gradient_boosting = run_experiment(
    name="Gradient Boosting",
    pipeline=Pipeline([
        ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),
        ("model", GradientBoostingClassifier(
            subsample=0.8, random_state=RANDOM_STATE))]),
    param_grid={"sampler__sampling_strategy": [0.1, 0.25],
                "model__n_estimators": [100, 300],
                "model__learning_rate": [0.05, 0.1],
                "model__max_depth": [2, 3]},
    techniques="undersampling + drempel")

#### 5.7.3 XGBoost: hyperparameter tuning
**Pipeline:** alleen `XGBClassifier` met `tree_method="hist"`. Bij de histogrammethode worden numerieke features in een beperkt aantal bakjes (bins) verdeeld. De boom zoekt splits op basis van deze indeling, waardoor minder splitpunten hoeven te worden onderzocht. Standaardiseren is niet nodig (XGBoost developers, z.d.-a).

**Technieken tegen imbalance:** (1) klasseweging via `scale_pos_weight`: fouten op stroke-patiënten tellen zwaarder mee in de loss; (2) beslisdrempel afstemmen.

**Vaste regularisatie:**

- `subsample=0.8`: iedere boom ziet een willekeurige 80% van de patiënten.
- `colsample_bytree=0.8`: iedere boom gebruikt een willekeurige selectie van ongeveer 80% van de features.
- `min_child_weight=5`: stelt een ondergrens aan de som van de tweede afgeleiden van de verliesfunctie (Hessians) in een kindknoop. Het is dus geen minimumaantal patiënten per blad. Een hogere waarde maakt splitsen moeilijker (XGBoost developers, z.d.-b).

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `n_estimators` | 200 / 400 | Aantal bomen dat na elkaar wordt toegevoegd |
| `learning_rate` | 0,05 / 0,1 | Hoe sterk iedere nieuwe boom de voorspelling bijstuurt |
| `max_depth` | 2 / 3 | Diepte van iedere boom; we beginnen met ondiepe bomen |
| `scale_pos_weight` | mild (≈ 8) / verhouding (≈ 64) | Relatief gewicht van een stroke-patiënt |

Een gebruikelijke eerste keuze voor `scale_pos_weight` is de verhouding tussen geen stroke en stroke, hier ongeveer 64 : 1. Dat geeft een vergelijkbare relatieve klasseweging als `class_weight="balanced"`, maar beide instellingen werken niet precies hetzelfde. We vergelijken dit met een mild gewicht: de wortel van die verhouding (≈ 8), gekozen als eenvoudige tweede mogelijkheid. Met die mildere weging krijgen stroke-patiënten minder extra nadruk. De cross-validation bepaalt welke optie beter scoort. Daarna stellen we de drempel af. Met 2 × 2 × 2 × 2 = 16 combinaties en 5 folds blijft het grid overzichtelijk.

(XGBoost developers, z.d.-b)

In [ ]:
# Verhouding geen stroke : stroke in de trainingsset (ongeveer 64 : 1)
neg_pos_ratio = (y_train == 0).sum() / y_train.sum()
mild_weight = round(np.sqrt(neg_pos_ratio), 1)  # mild klassegewicht (≈ 8)
print("scale_pos_weight: mild =", mild_weight,
      "verhouding =", round(neg_pos_ratio, 1))

# XGBoost met histogrammethode; vaste regularisatie via subsample,
# colsample_bytree en min_child_weight. n_jobs=1 per model.
xgboost_model = run_experiment(
    name="XGBoost",
    pipeline=Pipeline([
        ("model", XGBClassifier(
            tree_method="hist", subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, eval_metric="logloss",
            n_jobs=1, random_state=RANDOM_STATE))]),
    param_grid={"model__n_estimators": [200, 400],
                "model__learning_rate": [0.05, 0.1],
                "model__max_depth": [2, 3],
                "model__scale_pos_weight": [mild_weight,
                                            round(neg_pos_ratio, 1)]},
    techniques="class weights (scale_pos_weight) + drempel")

#### 5.7.4 Trainingstijd: sequentieel tegenover parallel
We vergelijken Random Forest, Gradient Boosting en XGBoost op dezelfde trainingsdata, met 300 bomen en een maximale diepte van 3. Random Forest en XGBoost meten we met één logische CPU en met alle beschikbare logische CPU's. Gradient Boosting heeft geen `n_jobs`-instelling. Per configuratie trainen we drie nieuwe modellen en nemen we de mediaan van de trainingstijden.

Voor deze meting gebruiken we geen undersampling of klassegewichten. De overige instellingen blijven standaard, behalve de histogrammethode van XGBoost. De data, het aantal bomen en de maximale diepte zijn dus gelijk, maar de algoritmen verrichten niet precies hetzelfde rekenwerk. De uitkomst is een praktische vergelijking en hangt ook af van de computer.

Random Forest kan onafhankelijke bomen parallel trainen. Bij boosting worden bomen na elkaar toegevoegd; XGBoost kan het werk binnen een boom parallel uitvoeren. De histogrammethode kan de zoektocht naar splits versnellen. Welke configuratie op onze computer het snelst is, bepalen we uit de daadwerkelijke metingen.

(scikit-learn developers, z.d.-b; XGBoost developers, z.d.-a)

In [ ]:
N_CORES = os.cpu_count() or 1  # aantal logische CPU's


def median_fit_time(model, X, y, n_runs=3):
    """Mediaan van de trainingstijd; iedere run traint een nieuw (gekloond) model."""
    times = []
    for _ in range(n_runs):
        fresh_model = clone(model)
        start = time.perf_counter()
        fresh_model.fit(X, y)
        times.append(time.perf_counter() - start)
    return np.median(times)


# Alle modellen: 300 bomen met diepte 3, verder standaardinstellingen.
# Een lijst houdt alle configuraties apart, ook wanneer N_CORES gelijk is aan 1.
timing_models = [
    ("Random Forest", 1, RandomForestClassifier(
        n_estimators=300, max_depth=3, n_jobs=1, random_state=RANDOM_STATE)),
    ("Random Forest", N_CORES, RandomForestClassifier(
        n_estimators=300, max_depth=3, n_jobs=N_CORES, random_state=RANDOM_STATE)),
    ("Gradient Boosting", 1, GradientBoostingClassifier(
        n_estimators=300, max_depth=3, random_state=RANDOM_STATE)),
    ("XGBoost", 1, XGBClassifier(
        n_estimators=300, max_depth=3, tree_method="hist", n_jobs=1,
        random_state=RANDOM_STATE)),
    ("XGBoost", N_CORES, XGBClassifier(
        n_estimators=300, max_depth=3, tree_method="hist", n_jobs=N_CORES,
        random_state=RANDOM_STATE)),
]

# Iedere configuratie op dezelfde trainingsset meten, snelste bovenaan
timing = pd.DataFrame(
    [{"model": name, "kernen": cores,
      "mediaan (s)": median_fit_time(model, X_train, y_train)}
     for name, cores, model in timing_models]
).sort_values("mediaan (s)").reset_index(drop=True)
timing["t.o.v. snelste"] = timing["mediaan (s)"] / timing["mediaan (s)"].min()

# Horizontale staafgrafiek, snelste bovenaan
labels = [f"{name} ({cores} {'kern' if cores == 1 else 'kernen'})"
          for name, cores in zip(timing["model"], timing["kernen"])]
seconds = timing["mediaan (s)"]
fig, ax = plt.subplots(figsize=(8, 3.5))
# Numerieke posities voorkomen dat gelijke labels op dezelfde balk terechtkomen.
positions = np.arange(len(timing))
ax.barh(positions, seconds.to_numpy()[::-1], color="#4C72B0")
ax.set_yticks(positions, labels=labels[::-1])
for y_pos, value in enumerate(seconds.to_numpy()[::-1]):
    ax.annotate(f"{value:.2f} s", (value, y_pos), xytext=(4, 0),
                textcoords="offset points", va="center", fontsize=9)
ax.set_xlim(0, seconds.max() * 1.15)  # ruimte voor de labels
ax.set(title="Trainingstijd met 300 bomen en diepte 3 (mediaan van 3 runs)",
       xlabel="Trainingstijd (seconden)")
print("Aantal patiënten:", len(y_train), "logische CPU's:", N_CORES)
plt.tight_layout()
plt.show()
timing.round(2)

### 5.8 Zelf samengesteld ensemble
We bekijken eerst hoe sterk de scores van de beschikbare losse modellen
samenhangen. Daarna combineren we Logistic Regression, Random Forest en
XGBoost met soft voting en vergelijken we het ensemble met deze modellen.

**Codebasis:** Anthropic (2026), zie de bronverantwoording AI hierboven.

#### 5.8.1 Overeenkomst tussen de losse modellen
We vergelijken de out-of-fold scores van de beschikbare losse modellen met Spearman-correlatie. Deze maat kijkt naar de rangorde van patiënten, zodat kansen en SVM-afstanden naast elkaar kunnen worden vergeleken. Een hoge correlatie betekent dat modellen patiënten ongeveer in dezelfde risicovolgorde zetten; het bewijst niet dat ze precies dezelfde classificatiefouten maken.

De gedeelde klasse bewaart de out-of-fold scores niet. Daarom berekenen we die hier met de folds van ieder experiment en bewaren we ze voor hergebruik. Als een model of de gebruikte trainingsdata wordt vervangen, worden de betreffende scores opnieuw berekend. KNN wordt meegenomen zodra dat model beschikbaar is. Het eigen voting-ensemble blijft buiten deze vergelijking van losse modellen.

De cache herkent vervanging van het model, de data of de folds. Als je bestaande objecten ter plekke wijzigt, maak je de cache eerst leeg met `oof_cache = {}` en voer je de scorecel opnieuw uit. Bij constante scores kan een correlatie niet worden bepaald; die vakjes blijven leeg.

(pandas development team, z.d.; SciPy developers, z.d.)

In [ ]:
# Out-of-fold scores bewaren; opnieuw getunede modellen krijgen nieuwe scores.
if "oof_cache" not in globals():
    oof_cache = {}


def oof_score(experiment):
    """Out-of-fold score van het getunede model, met dezelfde folds als tune_threshold()."""
    model = experiment.best_model
    method = ("predict_proba" if hasattr(model, "predict_proba")
              else "decision_function")  # SVM: afstand tot het hypervlak
    scores = cross_val_predict(clone(model), data["X_train"], data["y_train"],
                               cv=experiment.cv, method=method, n_jobs=N_JOBS)
    return scores[:, 1] if method == "predict_proba" else scores


# Alleen losse modellen; KNN doet mee zodra het in results staat.
model_names = ["Logistic Regression", "KNN", "SVM", "Decision Tree",
               "Random Forest", "Gradient Boosting", "XGBoost"]
experiments = {name: results.experiments[name] for name in model_names
               if name in results.experiments}
for name, experiment in experiments.items():
    context = (experiment.best_model, data["X_train"], data["y_train"], experiment.cv)
    cached = oof_cache.get(name)
    if cached is None or any(old is not new for old, new in zip(cached[0], context)):
        start = time.perf_counter()
        oof_cache[name] = (context, oof_score(experiment))
        print(name, ": out-of-fold scores berekend in",
              round(time.perf_counter() - start), "seconden")

oof_scores = {name: oof_cache[name][1] for name in experiments}
oof_frame = pd.DataFrame(oof_scores, index=data["y_train"].index)
print("Modellen:", ", ".join(oof_frame.columns))

In [ ]:
# Spearman vergelijkt de rangorde van kansen en SVM-scores.
oof_corr = oof_frame.corr(method="spearman")

fig, ax = plt.subplots(figsize=(1.1 * len(oof_corr) + 3, 0.9 * len(oof_corr) + 2))
sns.heatmap(oof_corr, annot=True, fmt=".2f", cmap="coolwarm", square=True,
            vmin=-1, vmax=1, center=0, cbar_kws={"label": "Spearman ρ"},
            ax=ax)
ax.set_title("Spearman-correlatie tussen de out-of-fold scores")
plt.tight_layout()
plt.show()

# Paren van minst naar meest vergelijkbaar (handig voor het ensemble)
upper = oof_corr.where(np.triu(np.ones(oof_corr.shape, dtype=bool), k=1))
upper.stack().dropna().sort_values().rename("Spearman ρ").to_frame().round(3)

#### 5.8.2 Eigen ensemble: soft voting
Een **soft-voting ensemble** combineert meerdere modellen door hun voorspelde kansen te middelen. Met gewichten $w_m$ wordt de ensemblescore:

$$\hat{p}_{\text{ensemble}} = \frac{\sum_m w_m \, \hat{p}_m}{\sum_m w_m}$$

Daarna kiezen we, net als bij de andere modellen, één drempel op deze gemiddelde score. Een ensemble kan helpen als de modellen verschillende fouten maken: waar het ene model een patiënt mist, kan een ander hem wel oppikken.

We combineren drie modellen die op een verschillende manier werken:

- **Logistic Regression**: een lineair model;
- **Random Forest**: een gemiddelde van onafhankelijke bomen (bagging);
- **XGBoost**: een model dat opeenvolgend bomen toevoegt om eerdere fouten te verkleinen (boosting).

Ieder model wordt overgenomen als kopie van zijn beste pipeline, met de getunede hyperparameters en de eigen technieken tegen de imbalance. We tunen alleen de gewichten:

| Gewichten [LR, RF, XGB] | Betekenis |
|---|---|
| [1, 1, 1] | Alle modellen even zwaar |
| [2, 1, 1] | Logistic Regression dubbel |
| [1, 1, 2] | XGBoost dubbel |

Door de verschillende klassegewichten zijn de voorspelde kansen niet noodzakelijk goed gekalibreerd. De modellen kunnen daardoor verschillende kansschalen hebben, wat het gemiddelde beïnvloedt. Een hogere voorspelde kans betekent niet dat een model automatisch een groter stemgewicht krijgt. We kiezen de gewichten met cross-validation en bepalen daarna de beslisdrempel uit de out-of-fold scores. De ensemblescore behandelen we als risicoscore, niet als gegarandeerd gekalibreerde kans.

De hyperparameters van de losse modellen zijn al op de trainingsset geselecteerd. Daardoor kan de CV-F1 van het ensemble optimistisch zijn. De aparte validatieset gebruiken we niet om de gewichten of de drempel te kiezen; daar vergelijken we de uiteindelijke modellen. Met drie gewichtsverdelingen zijn er 15 CV-trainingen, één refit, vijf trainingen voor de drempelkeuze en één voor het submissionbestand: 22 ensembletrainingen, elk met drie basismodellen.

(scikit-learn developers, z.d.-b, z.d.-d)

In [ ]:
# Kopieën (ongetraind) van de beste pipelines, met hun eigen instellingen en
# technieken tegen de imbalance (class weights, balanced_subsample, scale_pos_weight)
base_models = {"lr": log_reg, "rf": random_forest, "xgb": xgboost_model}
voting_estimators = [(short, clone(experiment.best_model))
                     for short, experiment in base_models.items()]

# Soft voting: gewogen gemiddelde van de kansen; alleen de gewichten worden getuned.
# Volgorde van de gewichten: [LR, RF, XGB]
ensemble = run_experiment(
    name="Eigen ensemble",
    pipeline=Pipeline([
        ("model", VotingClassifier(estimators=voting_estimators,
                                   voting="soft", n_jobs=1))]),
    param_grid={"model__weights": [[1, 1, 1],    # alles even zwaar
                                   [2, 1, 1],    # Logistic Regression dubbel
                                   [1, 1, 2]]},  # XGBoost dubbel
    techniques="technieken losse modellen + drempel")

In [ ]:
# Kleine vergelijking: ensemble tegenover de drie losse modellen
compare_names = [experiment.name for experiment in base_models.values()] + [ensemble.name]
compare_columns = ["CV-F1", "validatie-F1", "precision", "recall",
                   "ROC-AUC", "PR-AUC", "drempel", "beste hyperparameters"]
results.to_frame().loc[compare_names, compare_columns].round(3)